In [149]:
import pandas as pd
import re
pattern=r"(?:January|February|March|April|May|June|July|August|September|October|November|December)-\d{1,2}-\d{4}"

# Match Data Processing

In [151]:
match_name='Arsenal-Brighton-and-Hove-Albion-August-31-2024-Premier-League'
data_path='../data/premier_league/2024-25'
home_df=pd.read_csv(f'{data_path}/{match_name}_home_player_stat.csv')
away_df=pd.read_csv(f'{data_path}/{match_name}_away_player_stat.csv')
match_df=pd.read_csv(f'{data_path}/{match_name}_match_stat.csv', index_col=0)

In [33]:
match_df

,possession,passes,pass_accuracy,shots,shots_on_target
Arsenal,36,296,73,11,7
Brighton,64,518,83,22,4


In [34]:
target_columns=[
    'performance_gls',
    'performance_sh',
    'performance_sot',
    'performance_crdy',
    'corner_kicks_in',
    'corner_kicks_out',
    'corner_kicks_str',
]

In [35]:
assert 'Players' in home_df.iloc[0, 0]
assert 'Players' in away_df.iloc[0, 0]
home_ind=home_df.iloc[0, 0]
away_ind=away_df.iloc[0, 0]
home_df.set_index('player', inplace=True)
away_df.set_index('player', inplace=True)
targets={
    'home_goals': home_df.loc[home_ind, 'performance_gls'],
    'away_goals': away_df.loc[away_ind, 'performance_gls'],
    'home_corners': home_df.loc[home_ind, 'corner_kicks_in']+home_df.loc[home_ind, 'corner_kicks_out']+home_df.loc[home_ind, 'corner_kicks_str'],
    'away_corners': away_df.loc[away_ind, 'corner_kicks_in']+away_df.loc[away_ind, 'corner_kicks_out']+away_df.loc[away_ind, 'corner_kicks_str'],
    'home_cards': home_df.loc[home_ind, 'performance_crdy'],
    'away_cards': away_df.loc[away_ind, 'performance_crdy'],
    'home_shots': home_df.loc[home_ind, 'performance_sh'],
    'away_shots': away_df.loc[away_ind, 'performance_sh'],
    'home_sots': home_df.loc[home_ind, 'performance_sot'],
    'away_sots': away_df.loc[away_ind, 'performance_sot'],
}

In [37]:
targets['total_goals']=targets['home_goals']+targets['away_goals']
targets['total_corners']=targets['home_corners']+targets['away_corners']
targets['total_cards']=targets['home_cards']+targets['away_cards']
targets['total_shots']=targets['home_shots']+targets['away_shots']
targets['total_sots']=targets['home_sots']+targets['away_sots']

In [44]:
targets_df=pd.DataFrame(pd.Series(targets)).transpose()

In [45]:
targets_df

,home_goals,away_goals,home_corners,away_corners,home_cards,away_cards,home_shots,away_shots,home_sots,away_sots,total_goals,total_corners,total_cards,total_shots,total_sots
0,1,1,3,5,5,2,11,22,7,4,2,8,7,33,11


In [119]:
def process_match_target_var(home_df, away_df, match_df, match_name):
    assert 'Players' in home_df.iloc[0, 0]
    assert 'Players' in away_df.iloc[0, 0]
    home_ind=home_df.iloc[0, 0]
    away_ind=away_df.iloc[0, 0]
    home_df=home_df.set_index('player')
    away_df=away_df.set_index('player')
    targets={
        'home_goals': home_df.loc[home_ind, 'performance_gls'],
        'away_goals': away_df.loc[away_ind, 'performance_gls'],
        'home_corners': home_df.loc[home_ind, 'corner_kicks_in']+home_df.loc[home_ind, 'corner_kicks_out']+home_df.loc[home_ind, 'corner_kicks_str'],
        'away_corners': away_df.loc[away_ind, 'corner_kicks_in']+away_df.loc[away_ind, 'corner_kicks_out']+away_df.loc[away_ind, 'corner_kicks_str'],
        'home_cards': home_df.loc[home_ind, 'performance_crdy'],
        'away_cards': away_df.loc[away_ind, 'performance_crdy'],
        'home_shots': home_df.loc[home_ind, 'performance_sh'],
        'away_shots': away_df.loc[away_ind, 'performance_sh'],
        'home_sots': home_df.loc[home_ind, 'performance_sot'],
        'away_sots': away_df.loc[away_ind, 'performance_sot'],
    }
    targets_df=pd.DataFrame(pd.Series(targets)).transpose()
    targets_df['home']=match_df.index.tolist()[0]
    targets_df['away']=match_df.index.tolist()[1]
    match_date=re.findall(pattern, match_name, re.IGNORECASE)[0]
    targets_df['date']=[match_date]
    return targets_df

In [121]:
process_match_target_var(home_df, away_df, match_df,match_name)

,home_goals,away_goals,home_corners,away_corners,home_cards,away_cards,home_shots,away_shots,home_sots,away_sots,home,away,date
0,3,1,5,4,2,5,14,9,6,3,Bournemouth,Southampton,September-30-2024


# Generate all target data for all matches

In [122]:
seasons=[
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
]
data_path='../data/premier_league/'
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_matches=pd.read_csv(f'{season_path}/all_matches.csv').iloc[:,1].tolist()
    target_df=[]
    for match_name in all_matches:
        home_df=pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
        away_df=pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
        match_df=pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
        target_df.append(process_match_target_var(home_df, away_df, match_df, match_name))
    target_df=pd.concat(target_df, axis=0)
    target_df.to_csv(f'{season_path}/all_target_df.csv', index=False)
    

# Other Statistics

In [143]:
target_columns=[
    'player',
    'performance_ast',
    'performance_gls',
    'performance_sh',
    'performance_sot',
    'performance_crdy',
    'corner_kicks_in',
    'corner_kicks_out',
    'corner_kicks_str',
    'player_ast',
]

In [159]:
def process_match_other_var(home_df, away_df, match_df, match_name):
    assert 'Players' in home_df.iloc[0, 0]
    assert 'Players' in away_df.iloc[0, 0]
    home_df=home_df.drop(columns=target_columns).rename(columns=lambda x: f'home_{x}').head(1)
    away_df=away_df.drop(columns=target_columns).rename(columns=lambda x: f'away_{x}').head(1)
    data_df=pd.concat([home_df, away_df], axis=1)
    data_df['home']=[match_df.index.tolist()[0]]
    data_df['away']=[match_df.index.tolist()[1]]
    match_date=re.findall(pattern, match_name, re.IGNORECASE)[0]
    data_df['date']=[match_date]
    return data_df

In [160]:
seasons=[
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
]
data_path='../data/premier_league/'
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_matches=pd.read_csv(f'{season_path}/all_matches.csv').iloc[:,1].tolist()
    data_df=[]
    for match_name in all_matches:
        home_df=pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
        away_df=pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
        match_df=pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
        data_df.append(process_match_other_var(home_df, away_df, match_df, match_name))
    data_df=pd.concat(data_df, axis=0)
    data_df.to_csv(f'{season_path}/all_data_df.csv', index=False)